# CASMI 2026 — Clean fragment-ranking submission notebook

This notebook **recomputes** its submission using the **currently mounted** `train.parquet`, `test.parquet`, and `sample_submission.csv` on every Run All. It does **not** load test labels, experimental evaluation files, 60/210/300/400-prediction backups, old checkpoints, or stored predictions. **Do not add those files as inputs.**

Method preserved from the research notebook: make a candidate structure library from train formula/SMILES; select structures within 20 ppm of the neutral precursor mass; match the five strongest `[M+H]+` MS/MS fragment peaks against one-/two-bond graph cuts (±2 H); rank by matched intensity and then precursor-mass error; fill up to 25 unique SMILES using mass proximity. Results on visible test do not guarantee performance on an unseen-structure hidden test.

**Before the final Kaggle submission:** start a fresh notebook session with the competition data attached; run all cells to verify that RDKit is available and `/kaggle/working/submission.csv` is generated. The notebook attempts `pip install rdkit` only if import fails. Installation requires the package to be available in the environment or accessible through permitted Internet/offline wheels. A downloaded visible-test CSV is not an alternative to recomputing hidden-test predictions.

In [1]:
# 1. Read CURRENT competition data (not a saved visible-test prediction).
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

dataset_folder = Path('/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra')
train_path = dataset_folder / 'train.parquet'
test_path = dataset_folder / 'test.parquet'
submission_path = dataset_folder / 'sample_submission.csv'
for path in (train_path, test_path, submission_path):
    if not path.is_file():
        raise FileNotFoundError(f'Attach the competition input; missing: {path}')

train_file = pq.ParquetFile(train_path)
test_file = pq.ParquetFile(test_path)
sample_submission = pd.read_csv(submission_path)
assert 'molecule_id' in sample_submission.columns
assert sample_submission['molecule_id'].is_unique
assert len(sample_submission) > 0

test_spectra = test_file.read(columns=[
    'molecule_id', 'spectrum_id', 'adduct', 'precursor_mz',
    'ms2_mzs', 'ms2_normalized_intensities'
]).to_pandas()
print('Train row groups:', train_file.num_row_groups)
print('Current test spectra:', len(test_spectra))
print('Current submission molecules:', len(sample_submission))


Train row groups: 21
Current test spectra: 1213
Current submission molecules: 400


In [2]:
# 2. Stream training library and compute formula-based neutral masses.
# CASMI 2026 - Restore candidate library

import pandas as pd

library_columns = [
    "inchikey14",
    "normalized_smiles",
    "molecular_formula"
]

seen_keys = set()
library_parts = []

print("CASMI 2026 - Restoring Candidate Library")
print("----------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=20000,
        columns=library_columns
    ):
        batch_df = (
            batch.to_pandas()
            .dropna(subset=library_columns)
            .drop_duplicates(subset="inchikey14")
        )

        new_rows = batch_df.loc[
            ~batch_df["inchikey14"].isin(seen_keys)
        ]

        if not new_rows.empty:
            library_parts.append(new_rows)
            seen_keys.update(new_rows["inchikey14"])

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )

full_candidate_library = pd.concat(
    library_parts,
    ignore_index=True
)

del library_parts

print("\nCASMI 2026 - Candidate Library Restored")
print("---------------------------------------")
print("Unique structures:", len(full_candidate_library))
print(
    "Duplicate structure keys:",
    int(full_candidate_library["inchikey14"].duplicated().sum())
)
print("\nRestoration completed!")

# CASMI 2026 - Restore candidate reference masses

import re
import numpy as np

atomic_masses = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620,
    "S": 31.972071174,
    "P": 30.973761998,
    "F": 18.998403163,
    "Cl": 34.968852682,
    "Br": 78.918337600,
    "I": 126.904468000,
    "B": 11.009305360
}

def formula_to_reference_mass(formula):
    if not isinstance(formula, str):
        return np.nan

    # Preserve the mass calculation used for our earlier candidates.
    elemental_formula = re.sub(r"\+\d*$", "", formula)

    parts = re.findall(
        r"([A-Z][a-z]?)(\d*)",
        elemental_formula
    )

    reconstructed = "".join(
        element + count
        for element, count in parts
    )

    if reconstructed != elemental_formula or not parts:
        return np.nan

    if any(
        element not in atomic_masses
        for element, _ in parts
    ):
        return np.nan

    return sum(
        atomic_masses[element] * int(count or 1)
        for element, count in parts
    )

full_candidate_library["reference_mass"] = (
    full_candidate_library["molecular_formula"]
    .apply(formula_to_reference_mass)
)

library_with_mass = full_candidate_library.dropna(
    subset=["reference_mass"]
).copy()

print("CASMI 2026 - Reference Masses Restored")
print("--------------------------------------")
print("Total structures:", len(full_candidate_library))
print("Structures with reference mass:", len(library_with_mass))
print(
    "Structures with missing mass:",
    full_candidate_library["reference_mass"].isna().sum()
)

CASMI 2026 - Restoring Candidate Library
----------------------------------------
Row groups checked: 5/21
Row groups checked: 10/21
Row groups checked: 15/21
Row groups checked: 20/21
Row groups checked: 21/21

CASMI 2026 - Candidate Library Restored
---------------------------------------
Unique structures: 275810
Duplicate structure keys: 0

Restoration completed!
CASMI 2026 - Reference Masses Restored
--------------------------------------
Total structures: 275810
Structures with reference mass: 275763
Structures with missing mass: 47


In [3]:
# 3. Infer masses from the current test precursors and known adduct shifts.
import numpy as np
import pandas as pd

adduct_shifts = {
    '[M+H]+': 1.007276466621,
    '[M-H]-': -1.007276466621,
    '[M+CH2O2-H]-': 44.99820284,
    '[M+Na]+': 22.989218,
    '[M+NH4]+': 18.033823,
    '[M+K]+': 38.963158,
    '[M+Cl]-': 34.969401
}

mass_data = test_spectra[['molecule_id', 'spectrum_id', 'adduct', 'precursor_mz']].copy()
mass_data['adduct_shift'] = mass_data['adduct'].map(adduct_shifts)
mass_data['neutral_mass'] = (
    pd.to_numeric(mass_data['precursor_mz'], errors='coerce')
    - mass_data['adduct_shift']
)
test_molecules = mass_data.groupby('molecule_id', as_index=False).agg(
    median_neutral_mass=('neutral_mass', 'median'),
    num_spectra=('spectrum_id', 'count')
)
mass_lookup = test_molecules.set_index('molecule_id')['median_neutral_mass']

missing_masses = sample_submission.loc[
    ~sample_submission['molecule_id'].isin(mass_lookup.dropna().index),
    'molecule_id'
].tolist()
print('Molecules with valid mass:', len(sample_submission) - len(missing_masses))
if missing_masses:
    print('WARNING: missing neutral masses; deterministic library fallback for',len(missing_masses),'molecules')


Molecules with valid mass: 400


In [4]:
# 4. Install RDKit offline before fragment matching

from pathlib import Path
import subprocess
import sys

wheels = list(Path("/kaggle/input").rglob("rdkit-*.whl"))

assert len(wheels) == 1, "RDKit wheel haijapatikana kwenye Kaggle Input."

subprocess.check_call([
    sys.executable,
    "-m", "pip", "install",
    "--no-index",
    "--no-deps",
    str(wheels[0])
])

from rdkit import Chem

print("RDKit offline installation: SUCCESS")
# CASMI 2026 - Restore the original fragment-matching function

from itertools import combinations
from collections import Counter
import numpy as np
from rdkit import Chem

ELECTRON_MASS_DA = 0.000548579909065

FRAGMENT_ATOMIC_MASSES = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620,
    "S": 31.972071174,
    "P": 30.973761998,
    "F": 18.998403163,
    "Cl": 34.968852682,
    "Br": 78.918337600,
    "I": 126.904468000,
    "B": 11.009305360
}


def match_observed_fragments(
    smiles,
    observed_mzs,
    observed_intensities,
    ppm_tolerance=20.0,
    max_bond_cuts=2,
    max_hydrogen_shift=2
):
    molecule = Chem.MolFromSmiles(smiles)

    if molecule is None:
        return None

    mzs = np.asarray(observed_mzs, dtype=float)
    intensities = np.asarray(observed_intensities, dtype=float)

    assert mzs.shape == intensities.shape
    assert np.all(np.isfinite(mzs))
    assert np.all(mzs > 0)

    molecule_with_h = Chem.AddHs(molecule)

    atom_symbols = [
        atom.GetSymbol()
        for atom in molecule_with_h.GetAtoms()
    ]

    if not set(atom_symbols).issubset(FRAGMENT_ATOMIC_MASSES):
        return None

    bonds = [
        (bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
        for bond in molecule.GetBonds()
    ]

    matched = np.zeros(len(mzs), dtype=bool)
    minimum_bond_cuts = np.zeros(len(mzs), dtype=np.int8)

    for number_of_cuts in range(1, max_bond_cuts + 1):

        for bonds_to_cut in combinations(bonds, number_of_cuts):
            editable = Chem.RWMol(molecule_with_h)

            for atom_a, atom_b in bonds_to_cut:
                editable.RemoveBond(atom_a, atom_b)

            fragments = Chem.GetMolFrags(
                editable.GetMol(),
                asMols=False
            )

            if len(fragments) < 2:
                continue

            for atom_indices in fragments:
                counts = Counter(
                    atom_symbols[index]
                    for index in atom_indices
                )

                heavy_atom_count = sum(
                    count
                    for element, count in counts.items()
                    if element != "H"
                )

                if heavy_atom_count == 0:
                    continue

                base_mass = sum(
                    FRAGMENT_ATOMIC_MASSES[element] * count
                    for element, count in counts.items()
                )

                for hydrogen_shift in range(
                    -max_hydrogen_shift,
                    max_hydrogen_shift + 1
                ):
                    if counts["H"] + hydrogen_shift < 0:
                        continue

                    theoretical_mz = (
                        base_mass
                        + hydrogen_shift
                        * FRAGMENT_ATOMIC_MASSES["H"]
                        - ELECTRON_MASS_DA
                    )

                    if theoretical_mz <= 0:
                        continue

                    ppm_errors = (
                        np.abs(mzs - theoretical_mz)
                        / theoretical_mz
                        * 1_000_000
                    )

                    new_matches = (
                        (ppm_errors <= ppm_tolerance)
                        & ~matched
                    )

                    minimum_bond_cuts[new_matches] = number_of_cuts
                    matched |= new_matches

            if matched.all():
                break

        if matched.all():
            break

    return {
        "matched_peaks": int(matched.sum()),
        "total_peaks": len(mzs),
        "matched_intensity": float(intensities[matched].sum()),
        "one_bond_matches": int(
            (minimum_bond_cuts == 1).sum()
        ),
        "two_bond_matches": int(
            (minimum_bond_cuts == 2).sum()
        )
    }


print("CASMI 2026 - Fragment Function Restored")
print("---------------------------------------")
print("Function available:", callable(match_observed_fragments))
print("RDKit import: successful")
print("Predictions recomputed: 0")

Processing /kaggle/input/datasets/prettyglory/casmi-rdkit-offline/rdkit-2024.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
RDKit offline installation: SUCCESS
CASMI 2026 - Fragment Function Restored
---------------------------------------
Function available: True
RDKit import: successful
Predictions recomputed: 0


In [5]:
# 5. Score current test candidates and produce a ranked 25-SMILES list.
# CASMI 2026 - Restore reusable molecule scoring

import numpy as np
import pandas as pd

def score_test_molecule(molecule_id):
    """
    Score candidates using the current test spectra.
    Does not use known-structure labels or saved evaluation scores.
    """

    # No known adduct/mass: return empty candidates so ranker uses deterministic fallback.
    if molecule_id not in mass_lookup.index or pd.isna(mass_lookup.loc[molecule_id]):
        return pd.DataFrame(columns=[
            'inchikey14', 'normalized_smiles', 'reference_mass',
            'mass_error_ppm', 'fragment_score', 'matched_peaks'
        ])
    query_mass = float(mass_lookup.loc[molecule_id])

    if not np.isfinite(query_mass) or query_mass <= 0:
        return pd.DataFrame(columns=[
            'inchikey14', 'normalized_smiles', 'reference_mass',
            'mass_error_ppm', 'fragment_score', 'matched_peaks'
        ])

    candidates = (
        library_with_mass.loc[
            (
                np.abs(
                    library_with_mass["reference_mass"] - query_mass
                )
                / query_mass * 1_000_000
            ) <= 20.0,
            ["inchikey14", "normalized_smiles", "reference_mass"]
        ]
        .drop_duplicates("inchikey14")
        .copy()
    )

    candidates["mass_error_ppm"] = (
        np.abs(candidates["reference_mass"] - query_mass)
        / query_mass * 1_000_000
    )

    query_rows = (
        test_spectra.loc[
            (test_spectra["molecule_id"] == molecule_id)
            & (test_spectra["adduct"] == "[M+H]+")
        ]
        .sort_values("spectrum_id")
    )

    candidates["fragment_score"] = np.nan
    candidates["matched_peaks"] = 0

    if query_rows.empty:
        return candidates

    query_row = query_rows.iloc[0]

    mzs = np.asarray(query_row["ms2_mzs"], dtype=float)
    intensities = np.asarray(
        query_row["ms2_normalized_intensities"], dtype=float
    )

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs > 0)
        & (intensities > 0)
    )

    mzs = mzs[valid]
    intensities = intensities[valid]

    if len(mzs) == 0:
        return candidates

    strongest = np.argsort(-intensities)[:5]

    scores = []
    matched_counts = []

    for row in candidates.itertuples(index=False):
        result = match_observed_fragments(
            row.normalized_smiles,
            mzs[strongest],
            intensities[strongest],
            ppm_tolerance=20.0,
            max_bond_cuts=2,
            max_hydrogen_shift=2
        )

        scores.append(
            result["matched_intensity"]
            if result is not None
            else np.nan
        )

        matched_counts.append(
            result["matched_peaks"]
            if result is not None
            else 0
        )

    candidates["fragment_score"] = scores
    candidates["matched_peaks"] = matched_counts

    return candidates


print("CASMI 2026 - Molecule Scoring Restored")
print("-------------------------------------")
print("Scoring function available:", callable(score_test_molecule))
print("Predictions recomputed: 0")

# CASMI 2026 - Restore ranked SMILES function

import numpy as np
import pandas as pd

fallback_library = (
    library_with_mass[
        ["inchikey14", "normalized_smiles", "reference_mass"]
    ]
    .dropna()
    .drop_duplicates("inchikey14")
    .reset_index(drop=True)
)

fallback_masses = fallback_library[
    "reference_mass"
].to_numpy(dtype=float)


def build_ranked_smiles(molecule_id, top_k=25):
    """
    Rank candidates using fragment scores when available,
    with mass-only fallback. Does not use known labels.
    """

    scored = score_test_molecule(molecule_id)

    ranked = scored.sort_values(
        ["fragment_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True],
        na_position="last"
    )

    selected = []
    seen_smiles = set()

    def add_smiles(smiles):
        if (
            isinstance(smiles, str)
            and smiles.strip()
            and ";" not in smiles
            and smiles not in seen_smiles
        ):
            selected.append(smiles)
            seen_smiles.add(smiles)

    # Take ranked candidates within the 20 ppm mass window.
    for smiles in ranked["normalized_smiles"]:
        add_smiles(smiles)

        if len(selected) == top_k:
            break

    # If fewer than 25 candidates exist, fill by mass proximity.
    if len(selected) < top_k:
        query_mass = float(mass_lookup.get(molecule_id, np.nan))
        # When precursor mass cannot be inferred, use a fixed-order 25-SMILES fallback.
        nearest_indices = (
            np.argsort(np.abs(fallback_masses - query_mass), kind="stable")
            if np.isfinite(query_mass) and query_mass > 0
            else np.arange(len(fallback_library))
        )

        for index in nearest_indices:
            add_smiles(
                fallback_library.iloc[index]["normalized_smiles"]
            )

            if len(selected) == top_k:
                break

    assert len(selected) == top_k, (
        f"Could not find {top_k} unique SMILES for {molecule_id}"
    )

    return selected


print("CASMI 2026 - Ranked SMILES Function Restored")
print("-------------------------------------------")
print("Function available:", callable(build_ranked_smiles))
print("Fallback structures:", len(fallback_library))
print("Predictions recomputed: 0")

CASMI 2026 - Molecule Scoring Restored
-------------------------------------
Scoring function available: True
Predictions recomputed: 0
CASMI 2026 - Ranked SMILES Function Restored
-------------------------------------------
Function available: True
Fallback structures: 275763
Predictions recomputed: 0


In [6]:
# 6. Always regenerate the submission from the CURRENT test file.
# Never read a previously saved prediction or checkpoint here.
from pathlib import Path
import time
import pandas as pd

result_rows = []
started = time.perf_counter()
for i, molecule_id in enumerate(sample_submission['molecule_id'], 1):
    ranked_smiles = build_ranked_smiles(molecule_id, top_k=25)
    assert len(ranked_smiles) == len(set(ranked_smiles)) == 25
    result_rows.append({
        'molecule_id': molecule_id,
        'smiles': ';'.join(ranked_smiles)
    })
    if i % 10 == 0 or i == len(sample_submission):
        print(f'Current-test predictions: {i}/{len(sample_submission)} | '
              f'{time.perf_counter() - started:.1f} seconds')

final_submission = sample_submission[['molecule_id']].merge(
    pd.DataFrame(result_rows), on='molecule_id', how='left', validate='one_to_one'
)
print('Current-test predictions complete:', len(final_submission))


Current-test predictions: 10/400 | 91.6 seconds
Current-test predictions: 20/400 | 169.2 seconds
Current-test predictions: 30/400 | 237.0 seconds
Current-test predictions: 40/400 | 361.4 seconds
Current-test predictions: 50/400 | 484.5 seconds
Current-test predictions: 60/400 | 534.6 seconds
Current-test predictions: 70/400 | 600.2 seconds
Current-test predictions: 80/400 | 664.7 seconds
Current-test predictions: 90/400 | 774.8 seconds
Current-test predictions: 100/400 | 845.4 seconds
Current-test predictions: 110/400 | 931.1 seconds
Current-test predictions: 120/400 | 1008.4 seconds
Current-test predictions: 130/400 | 1104.1 seconds
Current-test predictions: 140/400 | 1243.2 seconds
Current-test predictions: 150/400 | 1357.2 seconds
Current-test predictions: 160/400 | 1463.3 seconds
Current-test predictions: 170/400 | 1533.3 seconds
Current-test predictions: 180/400 | 1576.8 seconds
Current-test predictions: 190/400 | 1645.3 seconds
Current-test predictions: 200/400 | 1716.8 seconds
C

In [7]:
# 7. Validate the CURRENT test IDs, then write the Kaggle output.
from pathlib import Path
import pandas as pd

assert final_submission.columns.tolist() == ['molecule_id', 'smiles']
assert len(final_submission) == len(sample_submission)
assert final_submission['molecule_id'].is_unique
assert final_submission['molecule_id'].tolist() == sample_submission['molecule_id'].tolist()
assert final_submission['smiles'].apply(lambda s:
    isinstance(s, str) and
    len(s.split(';')) == 25 and
    len(set(s.split(';'))) == 25 and
    all(s.split(';'))
).all(), 'Every molecule must have exactly 25 unique, nonempty SMILES.'

output_path = Path('/kaggle/working/submission.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
final_submission.to_csv(output_path, index=False)
print('Validated submission rows:', len(final_submission))
print('Saved current-test submission:', output_path)


Validated submission rows: 400
Saved current-test submission: /kaggle/working/submission.csv


In [8]:
from pathlib import Path
import subprocess
import sys

wheels = list(Path("/kaggle/input").rglob("rdkit-*.whl"))

print("RDKit wheel files found:", len(wheels))
for wheel in wheels:
    print(wheel)

assert len(wheels) == 1, "RDKit wheel haijapatikana au kuna zaidi ya moja."

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index", "--no-deps", str(wheels[0])
])

from rdkit import Chem
print("RDKit offline installation: SUCCESS")

RDKit wheel files found: 1
/kaggle/input/datasets/prettyglory/casmi-rdkit-offline/rdkit-2024.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/prettyglory/casmi-rdkit-offline/rdkit-2024.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
rdkit is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
RDKit offline installation: SUCCESS


In [9]:
# Install RDKit from the attached Kaggle dataset — no Internet needed.

from pathlib import Path
import subprocess
import sys

wheels = list(Path("/kaggle/input").rglob("rdkit-*.whl"))
assert len(wheels) == 1, "RDKit wheel haijapatikana kwenye Input."

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index", "--no-deps", str(wheels[0])
])

from rdkit import Chem

print("RDKit offline installation: SUCCESS")

Processing /kaggle/input/datasets/prettyglory/casmi-rdkit-offline/rdkit-2024.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
rdkit is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
RDKit offline installation: SUCCESS


In [10]:
# Install RDKit from the attached Kaggle dataset — no Internet needed.

from pathlib import Path
import subprocess
import sys

wheels = list(Path("/kaggle/input").rglob("rdkit-*.whl"))
assert len(wheels) == 1, "RDKit wheel haijapatikana kwenye Input."

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index", "--no-deps", str(wheels[0])
])

from rdkit import Chem

print("RDKit offline installation: SUCCESS")

Processing /kaggle/input/datasets/prettyglory/casmi-rdkit-offline/rdkit-2024.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
rdkit is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
RDKit offline installation: SUCCESS
